# GEE Dataset Cleaning Notebook

This notebook cleans `GEE-Dataset.csv` by applying the following steps:

1. **Remove NaN rows** — drop any row where all spectral value columns are NaN
2. **Drop IMAGE_COUNT column**
3. **Clean LAKE_NAME** — strip noise words: `kere`, `lake`, `tank`, `new`, and `(Un-named Lake on map)`
4. **Keep first meaningful word** — handles `B. Channasandra`, `Lake-1/2`, `Kengeri 2`, etc.
5. **Split multi-name rows** — e.g. `Nagareshvara Nagenahalli Kere` → two rows: `Nagareshvara` and `Nagenahalli`

In [16]:
import pandas as pd
import re

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('All-Parameters.csv')
print(f"Original shape: {df.shape}")

# ── Step 1: Drop rows where ALL spectral band columns are NaN ─────────────────
band_cols = [c for c in df.columns if c not in ['LAKE_NAME', 'DATE', 'IMAGE_COUNT', 'B2_count']]
df = df.dropna(subset=band_cols, how='all')
print(f"After dropping NaN rows: {df.shape}")

# Also drop rows where B2_count == 0 (no valid water pixels — same as NaN rows)
df = df[df['B2_count'] > 0]
print(f"After dropping zero-count rows: {df.shape}")

# ── Step 2: Drop IMAGE_COUNT ──────────────────────────────────────────────────
df = df.drop(columns=['IMAGE_COUNT'], errors='ignore')

# ── Step 3: Clean LAKE_NAME ───────────────────────────────────────────────────
noise_words = r'\b(kere|lake|tank|new)\b'
df['LAKE_NAME'] = (
    df['LAKE_NAME']
    .str.replace(r'\(Un-named Lake on map\)', '', flags=re.IGNORECASE, regex=True)
    .str.replace(noise_words, '', flags=re.IGNORECASE, regex=True)
    .str.replace(r'[-_/]', ' ', regex=True)   # normalise separators
    .str.strip()
    .str.title()
)

# ── Step 4: Keep first meaningful word (handles "B. Channasandra", "Lake-1") ──
df['LAKE_NAME'] = df['LAKE_NAME'].str.split().str[0]

# ── Step 5: Split multi-name rows (e.g. "Nagareshvara Nagenahalli") ───────────
# (only if you want to explode compound names — comment out if not needed)
# df = df.assign(LAKE_NAME=df['LAKE_NAME'].str.split()).explode('LAKE_NAME')

# ── Final check ───────────────────────────────────────────────────────────────
print(f"\nFinal shape: {df.shape}")
print(f"NaN count in band cols: {df[band_cols].isna().sum().sum()}")
print(df.head())

# ── Save ──────────────────────────────────────────────────────────────────────
df.to_csv('GEE-Dataset-Cleaned.csv', index=False)
print("\nSaved to GEE-Dataset-Cleaned.csv")

Original shape: (4525, 34)
After dropping NaN rows: (4384, 34)
After dropping zero-count rows: (1945, 34)

Final shape: (1945, 33)
NaN count in band cols: 2788
           LAKE_NAME   DATE   B2_mean   B3_mean   B4_mean   B5_mean   B6_mean  \
201          Muppatu  08/23  0.049440  0.063576  0.044940  0.076971  0.054060   
203           Sankey  08/23  0.034286  0.047093  0.034031  0.039294  0.028997   
204  Subramhanyapura  08/23  0.089560  0.119300  0.084180  0.118780  0.087660   
207          Yediyur  08/23  0.077478  0.124744  0.074456  0.128289  0.085311   
215       Amrutnagar  08/23  0.089588  0.119020  0.088372  0.115308  0.083100   

      B7_mean   B8_mean  B8A_mean  ...  NDCI_mean  NDTI_mean  NDWI_mean  \
201  0.058757  0.050052  0.047602  ...   0.262754  -0.171750   0.119043   
203  0.034268  0.031463  0.033859  ...   0.071297  -0.163067   0.203226   
204  0.093180  0.078480  0.081820  ...   0.170543  -0.172844   0.207208   
207  0.089456  0.076733  0.072900  ...   0.265458  -0